In [1]:
%pip install pytest-playwright
!playwright install

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install nest_asyncio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import asyncio
import sys
import nest_asyncio
nest_asyncio.apply()

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

from playwright.sync_api import sync_playwright

import io
from PIL import Image, UnidentifiedImageError

import os
import json
from pathlib import Path
import pandas as pd
import requests
import time

In [4]:
PATH = Path('./saved_pages')
CONCURRENCY = 1

In [5]:
df = pd.read_csv('canonical_ingredients.csv')
df.head()

,id,canonical_ingredient,variant_count
0,1,abalone,13
1,2,adzuki beans,23
2,3,agave,16
3,4,aioli,45
4,5,alfredo sauce,73


In [6]:
ingredients_list = []
for i, row in df.iterrows():
    words = row['canonical_ingredient'].split(' ')
    if len(words) > 1:
        ingredient = words[0]
        for word in words[1:]:
            ingredient = ingredient + '+' + word
        ingredients_list.append(ingredient)
    else:
        ingredients_list.append(row['canonical_ingredient'])
ingredients_list

['abalone',
 'adzuki+beans',
 'agave',
 'aioli',
 'alfredo+sauce',
 'all-purpose+flour',
 'allspice',
 'almond+butter',
 'almond+flour',
 'almonds',
 'amaranth',
 'amaretto',
 'american+cheese',
 'anchovy',
 'anchovy+paste',
 'andouille',
 'angel+hair+pasta',
 'anise',
 'annatto',
 'apple',
 'apple+cider+vinegar',
 'apple+juice',
 'apricot',
 'arrowroot',
 'artichoke',
 'artichoke+hearts',
 'arugula',
 'asparagus',
 'avocado',
 'avocado+oil',
 'bacon',
 'baguette',
 'baking+powder',
 'baking+soda',
 'balsamic+vinegar',
 'bamboo+shoots',
 'banana',
 'barbecue+sauce',
 'barley',
 'basa',
 'basil',
 'bass',
 'bay+leaf',
 'bean+sprouts',
 'beef+broth',
 'beef+roast',
 'beef+steak',
 'beer',
 'beet',
 'bell+pepper',
 'bison',
 'bitter+melon',
 'black+bean+sauce',
 'black+beans',
 'black+pepper',
 'black-eyed+peas',
 'blackberry',
 'blue+cheese',
 'blue+cheese+dressing',
 'blueberry',
 'bok+choy',
 'bone+broth',
 'bonito+flakes',
 'bouillon',
 'boysenberry',
 'brandy',
 'bratwurst',
 'brazil

In [7]:
urls = [f'https://duckduckgo.com/?q=%22{ingredient}+raw+ingredient%22&iar=images&t=h_' for ingredient in ingredients_list]
urls

['https://duckduckgo.com/?q=%22abalone+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22adzuki+beans+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22agave+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22aioli+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22alfredo+sauce+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22all-purpose+flour+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22allspice+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22almond+butter+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22almond+flour+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22almonds+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22amaranth+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22amaretto+raw+ingredient%22&iar=images&t=h_',
 'https://duckduckgo.com/?q=%22american+cheese+raw+ingredient%22&iar=

In [8]:
def check_count(ingredient):
    name = "progress_thumbs.txt"
    if not os.path.exists(name):
        with open(name, "w") as f:
            f.write(f"{ingredient} 0\n")
        return 0
    
    with open(name, "r") as f:
        lines = f.readlines()
    for line in lines:
        parts = line.split()
        if len(parts) < 2:
            continue
        i, c = parts[0], parts[1]
        if ingredient == i:
            return int(c)
    
    with open(name, "a") as f:
        f.write(f"{ingredient} 0\n")
    return 0

In [9]:
def update_count(ingredient, count):
    name = "progress_thumbs.txt"
    with open(name, "r") as f:
        lines = f.readlines()

    found = False
    new_lines = []
    for line in lines:
        parts = line.split()
        if len(parts) < 2:
            new_lines.append(line)
            continue
        i, c = parts[0], parts[1]
        if ingredient == i:
            new_lines.append(f"{ingredient} {count}\n")
            found = True
        else:
            new_lines.append(line)
    
    if not found:
        new_lines.append(f"{ingredient} {count}\n")
        
    with open(name, "w") as f:
        f.writelines(new_lines)

In [10]:
def save_page(browser, url, ingredient):
    page = browser.new_page()
    try:
        count = check_count(ingredient)
        if count is None:
            count = 0
        if count >= 250:
            return {"url": url, "status": "skipped", "count": count}

        page.goto(url, wait_until="networkidle", timeout=30000)
        
        for _ in range(6):
            page.mouse.wheel(0, 2000)
            page.wait_for_timeout(800)
        
        thumb_urls = page.eval_on_selector_all(          "img",
            """els => els
                .map(e => e.getAttribute('src') || e.getAttribute('data-src') || e.getAttribute('data-lazy-src'))
                .filter(u => u && !u.startsWith('data:'))""")
        thumb_urls = list(dict.fromkeys(thumb_urls))

        # Normalize scheme-less URLs and filter obvious non-photos
        normalized = []
        bad_ext = (".ico", ".svg", ".gif")
        bad_substrings = ("favicon", "logo", "sprite", "icon", "vector")

        for u in thumb_urls:
            if not u:
                continue
            if u.startswith("//"):
                u = "https:" + u
            ul = u.lower()
            if ul.endswith(bad_ext):
                continue
            if any(s in ul for s in bad_substrings):
                continue
            normalized.append(u)
        
        file_name = ingredient.replace("+", "_")[:150]
        page_dir = PATH / file_name
        page_dir.mkdir(parents=True, exist_ok=True)
        thumb_dir = page_dir / "thumbs"
        thumb_dir.mkdir(parents=True, exist_ok=True)
        
        MIN_BYTES = 10_000
        MIN_W, MIN_H = 200, 200
        
        downloaded = 0
        for img_url in normalized:
            if count >= 250:
                break
        
            try:
                r = requests.get(img_url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
                if r.status_code != 200:
        
                    continue
                ct = r.headers.get("Content-Type", "").lower()
                if not ct.startswith("image/"):
                    continue
                if ct in ("image/svg+xml", "image/x-icon", "image/vnd.microsoft.icon"):
                    continue
        
                data = r.content or b""
                if len(data) < MIN_BYTES:
                    continue
        
                # PIL validation: must decode, must be raster, must be big enough
                try:
                    im = Image.open(io.BytesIO(data))
                    im.load()
                except (UnidentifiedImageError, OSError):
                    continue
        
                # reject icons masquerading as images
                if im.format == "ICO":
                    continue
        
                w, h = im.size
                if w < MIN_W or h < MIN_H:
                    continue
        
                # save as JPEG (normalize formats + strip alpha)
                im = im.convert("RGB")
                out = thumb_dir / f"{count + 1:04d}.jpg"
                im.save(out, format="JPEG", quality=90, optimize=True)
        
                count += 1
                downloaded += 1
                time.sleep(0.1)
        
            except Exception:
                continue
        
        html = page.content()
        (page_dir / f"{ingredient}.html").write_text(html, encoding="utf-8")
        
        update_count(ingredient, count)
        print(f"✓ Saved: {url} ({downloaded} new, total {count}/250)")
        return {"url": url, "status": "ok", "path": str(page_dir), "count": count}
    
    except Exception as e:
        print(f"✗ Failed: {url} — {e}")
        return {"url": url, "status": "error", "error": str(e)}
    
    finally:
        try:
            update_count(ingredient, count)
        except Exception:
            pass
        page.close()

In [11]:
def save_all(urls, ingredients):
    PATH.mkdir(exist_ok=True)
    results = []

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        for url, ingredient in zip(urls, ingredients):
            result = save_page(browser, url, ingredient)
            results.append(result)
        browser.close()

    with open("results.json", "w") as f:
        json.dump(results, f, indent=2)

    failed = [r for r in results if r["status"] == "error"]
    print(f"\nDone. {len(results) - len(failed)}/{len(results)} succeeded.")
    return results

In [14]:
results = await asyncio.to_thread(save_all, urls, ingredients_list)

✓ Saved: https://duckduckgo.com/?q=%22andouille+raw+ingredient%22&iar=images&t=h_ (20 new, total 250/250)
✓ Saved: https://duckduckgo.com/?q=%22angel+hair+pasta+raw+ingredient%22&iar=images&t=h_ (13 new, total 250/250)
✓ Saved: https://duckduckgo.com/?q=%22apple+raw+ingredient%22&iar=images&t=h_ (3 new, total 250/250)
✓ Saved: https://duckduckgo.com/?q=%22bean+sprouts+raw+ingredient%22&iar=images&t=h_ (18 new, total 250/250)
✓ Saved: https://duckduckgo.com/?q=%22camembert+raw+ingredient%22&iar=images&t=h_ (8 new, total 250/250)
✓ Saved: https://duckduckgo.com/?q=%22canned+tomatoes+raw+ingredient%22&iar=images&t=h_ (75 new, total 250/250)
✓ Saved: https://duckduckgo.com/?q=%22cantaloupe+raw+ingredient%22&iar=images&t=h_ (3 new, total 250/250)
✓ Saved: https://duckduckgo.com/?q=%22capers+raw+ingredient%22&iar=images&t=h_ (76 new, total 250/250)
✓ Saved: https://duckduckgo.com/?q=%22chicken+drumstick+raw+ingredient%22&iar=images&t=h_ (48 new, total 250/250)
✓ Saved: https://duckduckgo.com

In [15]:
failed_indices = [i for i, r in enumerate(results) if r["status"] == "error"]
failed_urls = [urls[i] for i in failed_indices]
failed_ingredients = [ingredients_list[i] for i in failed_indices]
if failed_urls:
  print(f"Retrying {len(failed_urls)} failed URLs...")
  print("Failed ingredients:", failed_ingredients[:10])  # preview
  retry_results = await save_all(failed_urls)
else:
  print("No failures!")

No failures!
